In [42]:
import glob
import pandas as pd
import re
import ast
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from mlxtend.frequent_patterns import apriori , association_rules

In [43]:
# ── 1. รวมไฟล์รีวิว 5 ไฟล์ ──────────────────────────────────────────────────
review_files = [
    "reviews_0-250.csv",
    "reviews_250-500.csv",
    "reviews_500-750.csv",
    "reviews_750-1250.csv",
    "reviews_1250-end.csv",
]
df = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in review_files],
    ignore_index=True
)

In [44]:
df.head(3)

,Unnamed: 0,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
0,0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0
1,1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
2,2,5061282401,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0


In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1094411 entries, 0 to 1094410
Data columns (total 19 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   Unnamed: 0                1094411 non-null  int64  
 1   author_id                 1094411 non-null  object 
 2   rating                    1094411 non-null  int64  
 3   is_recommended            926423 non-null   float64
 4   helpfulness               532819 non-null   float64
 5   total_feedback_count      1094411 non-null  int64  
 6   total_neg_feedback_count  1094411 non-null  int64  
 7   total_pos_feedback_count  1094411 non-null  int64  
 8   submission_time           1094411 non-null  object 
 9   review_text               1092967 non-null  object 
 10  review_title              783757 non-null   object 
 11  skin_tone                 923872 non-null   object 
 12  eye_color                 884783 non-null   object 
 13  skin_type                 9

In [46]:
# ── 2. ลบ column ที่ไม่ใช้ ───────────────────────────────────────────────────
df.drop(columns=["Unnamed: 0","eye_color","hair_color"], errors="ignore", inplace=True)

# ── 3. แปลง data type ───────────────────────────────────────────────────────
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["price_usd"] = pd.to_numeric(df["price_usd"], errors="coerce")
df["submission_time"] = pd.to_datetime(df["submission_time"], errors="coerce")

In [47]:
print(df.columns.tolist())
print(df[["rating","price_usd","submission_time"]].dtypes)

['author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'skin_type', 'product_id', 'product_name', 'brand_name', 'price_usd']
rating                      int64
price_usd                 float64
submission_time    datetime64[ns]
dtype: object


In [48]:
# ── 4. ลบรีวิวที่ข้อความว่างหรือสั้นกว่า 15 ตัวอักษร ─────────────────────────
df["review_text"] = df["review_text"].fillna("")
before = len(df)
df = df[df["review_text"].str.len() >= 15].copy()
print(f"Dropped short reviews: {before - len(df):,} | Remaining: {len(df):,}")

# ── 5. ลบ HTML tags ──────────────────────────────────────────────────────────
def remove_html(text: str) -> str:
    return re.sub(r"<[^>]+>", " ", str(text)).strip()

df["review_text"] = df["review_text"].apply(remove_html)

Dropped short reviews: 1,445 | Remaining: 1,092,966


In [49]:
print(f"Rows: {len(df):,}")
print(f"HTML เหลือ: {df['review_text'].str.contains(r'<[^>]+>').sum()}")
print(df["review_text"].iloc[0][:200])

Rows: 1,092,966
HTML เหลือ: 0
I use this with the Nudestix “Citrus Clean Balm & Make-Up Melt“ to double cleanse and it has completely changed my skin (for the better). The make-up melt is oil based and removes all of your makeup s


In [50]:
# ── 6. Standardize skin_type ────────────────────────────────────────────────
VALID_SKIN = {"combination", "dry", "oily", "normal"}

def standardize_skin(val):
    if pd.isna(val):
        return "unknown"
    v = str(val).strip().lower()
    return v if v in VALID_SKIN else "unknown"

df["skin_type"] = df["skin_type"].apply(standardize_skin)
print(df["skin_type"].value_counts())

# ── 7. ลบรีวิวซ้ำ — เก็บครั้งแรก ────────────────────────────────────────────
before = len(df)
df.sort_values("submission_time", inplace=True)
df.drop_duplicates(subset=["author_id", "product_id"], keep="first", inplace=True)
print(f"Dropped duplicates: {before - len(df):,} | Remaining: {len(df):,}")

skin_type
combination    543777
dry            185675
normal         131687
oily           120309
unknown        111518
Name: count, dtype: int64
Dropped duplicates: 5,517 | Remaining: 1,087,449


In [51]:
# ── 8. Balanced sampling ─────────────────────────────────────────────────────
# กำหนด label จาก rating
def label_sentiment(r):
    if r >= 4:   return "positive"
    if r <= 2:   return "negative"
    return "neutral"

df["sentiment"] = df["rating"].apply(label_sentiment)

def stratified_sample(group_df, n):
    """สุ่ม n แถวโดยรักษาสัดส่วน skin_type"""
    if len(group_df) <= n:
        return group_df
    skin_counts = group_df["skin_type"].value_counts(normalize=True)
    parts = []
    for skin, ratio in skin_counts.items():
        k = max(1, round(ratio * n))
        sub = group_df[group_df["skin_type"] == skin]
        parts.append(sub.sample(min(k, len(sub)), random_state=42))
    sampled = pd.concat(parts)
    # ปรับให้ได้ n พอดี (กรณี rounding)
    if len(sampled) > n:
        sampled = sampled.sample(n, random_state=42)
    return sampled

pos  = stratified_sample(df[df["sentiment"] == "positive"], 15_000)
neg  = df[df["sentiment"] == "negative"]          # เก็บทั้งหมด
neu  = stratified_sample(df[df["sentiment"] == "neutral"], 3_000)

df_clean = pd.concat([pos, neg, neu], ignore_index=True)
df_clean.drop(columns=["sentiment"], inplace=True)
print(df_clean["rating"].value_counts().sort_index())
print(f"\nFinal reviews shape: {df_clean.shape}")

df_clean.to_csv("sephora_reviews_clean.csv", index=False)
print("✅ Saved: sephora_reviews_clean.csv")

rating
1    60858
2    52784
3     3000
4     3316
5    11684
Name: count, dtype: int64

Final reviews shape: (131642, 16)
✅ Saved: sephora_reviews_clean.csv


In [52]:
df_clean.head(3)

,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,skin_type,product_id,product_name,brand_name,price_usd
0,1333774242,5,1.0,NaN,0,0,0,2020-04-09,Great sunscreen! You definitely get what you p...,NaN,lightMedium,combination,P456582,Invisible Physical Defense Mineral Sunscreen S...,Dermalogica,48.0
1,9629781860,5,1.0,NaN,0,0,0,2017-09-23,I got a sample of this product after an associ...,Holy grail for adult acne!,light,combination,P474078,Mini Goodbye Acne AHA/BHA Acne Clearing Gel F...,Peter Thomas Roth,20.0
2,9245108514,5,1.0,NaN,0,0,0,2017-09-01,this sunscreen is amazing. even though it come...,best sunscreen ever,light,combination,P454384,Mini PLAY Everyday Lotion SPF 50 with Sunflowe...,Supergoop!,22.0


In [53]:
df_clean.info() #ดูโครงสร้างข้อมูล

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131642 entries, 0 to 131641
Data columns (total 16 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   author_id                 131642 non-null  object        
 1   rating                    131642 non-null  int64         
 2   is_recommended            112611 non-null  float64       
 3   helpfulness               99782 non-null   float64       
 4   total_feedback_count      131642 non-null  int64         
 5   total_neg_feedback_count  131642 non-null  int64         
 6   total_pos_feedback_count  131642 non-null  int64         
 7   submission_time           131642 non-null  datetime64[ns]
 8   review_text               131642 non-null  object        
 9   review_title              97837 non-null   object        
 10  skin_tone                 110720 non-null  object        
 11  skin_type                 131642 non-null  object        
 12  pr

In [54]:
df_clean.isnull().sum() #เช็ค missing values

author_id                       0
rating                          0
is_recommended              19031
helpfulness                 31860
total_feedback_count            0
total_neg_feedback_count        0
total_pos_feedback_count        0
submission_time                 0
review_text                     0
review_title                33805
skin_tone                   20922
skin_type                       0
product_id                      0
product_name                    0
brand_name                      0
price_usd                       0
dtype: int64

In [55]:
df_clean.duplicated().sum() #เช็คข้อมูลซ้ำ

np.int64(0)

In [56]:
# ── โหลดข้อมูลสินค้า ─────────────────────────────────────────────────────────
prod = pd.read_csv("product_info.csv")
print(f"Products loaded: {len(prod):,} | Columns: {prod.columns.tolist()}")

# ── 1. ลบ column ที่ไม่ใช้ ───────────────────────────────────────────────────
drop_prod = [c for c in prod.columns if any(kw in c.lower() for kw in
             ["sale_price", "value_price", "variation", "kit"])]
print("Dropping:", drop_prod)
prod.drop(columns=drop_prod, inplace=True)

# ── 2. ลบ product_id ซ้ำ ─────────────────────────────────────────────────────
before = len(prod)
prod.drop_duplicates(subset=["product_id"], keep="first", inplace=True)
print(f"Duplicate products removed: {before - len(prod):,}")

# ── 3. แปลง data type ───────────────────────────────────────────────────────
prod["rating"]      = pd.to_numeric(prod["rating"],      errors="coerce")
prod["price_usd"]   = pd.to_numeric(prod["price_usd"],   errors="coerce")
prod["loves_count"] = pd.to_numeric(prod["loves_count"], errors="coerce")

# ── 4. แปลง binary columns เป็น 0/1 ─────────────────────────────────────────
binary_cols = [c for c in ["limited_edition", "new", "online_only",
                            "exclusive", "out_of_stock"] if c in prod.columns]
for col in binary_cols:
    prod[col] = prod[col].map(lambda x: 1 if str(x).strip().lower()
                               in ("1", "true", "yes") else 0)

# ── 5. Fill missing categories ───────────────────────────────────────────────
for col in ["secondary_category", "tertiary_category"]:
    if col in prod.columns:
        prod[col] = prod[col].fillna("Unknown")

# ── 6. แปลง ingredients string ───────────────────────────────────────────────
import ast

def parse_ingredients(val):
    if pd.isna(val) or str(val).strip() == "":
        return ""
    text = str(val).strip()
    # กรณีเป็น list string เช่น "['Water', 'Glycerin']"
    if text.startswith("["):
        try:
            items = ast.literal_eval(text)
            return ", ".join(str(i).strip() for i in items)
        except Exception:
            pass
    return text  # คืนค่าเดิมถ้าไม่ใช่ list string

if "ingredients" in prod.columns:
    prod["ingredients"] = prod["ingredients"].apply(parse_ingredients)

prod.to_csv("sephora_products_clean.csv", index=False)
print(f"✅ Saved: sephora_products_clean.csv | Shape: {prod.shape}")

Products loaded: 8,494 | Columns: ['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price']
Dropping: ['variation_type', 'variation_value', 'variation_desc', 'value_price_usd', 'sale_price_usd']
Duplicate products removed: 0
✅ Saved: sephora_products_clean.csv | Shape: (8494, 22)


In [57]:
prod.head(3)

,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,ingredients,price_usd,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,NaN,"Capri Eau de Parfum:, Alcohol Denat. (SD Alcoh...",35.0,...,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,"Alcohol Denat. (SD Alcohol 39C), Parfum (Fragr...",195.0,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,85.0,30.0
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.2500,16.0,3.4 oz/ 100 mL,"Alcohol Denat. (SD Alcohol 39C), Parfum (Fragr...",195.0,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0


In [58]:
prod.info() #ดูโครงสร้างข้อมูล

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8494 entries, 0 to 8493
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_id          8494 non-null   object 
 1   product_name        8494 non-null   object 
 2   brand_id            8494 non-null   int64  
 3   brand_name          8494 non-null   object 
 4   loves_count         8494 non-null   int64  
 5   rating              8216 non-null   float64
 6   reviews             8216 non-null   float64
 7   size                6863 non-null   object 
 8   ingredients         8494 non-null   object 
 9   price_usd           8494 non-null   float64
 10  limited_edition     8494 non-null   int64  
 11  new                 8494 non-null   int64  
 12  online_only         8494 non-null   int64  
 13  out_of_stock        8494 non-null   int64  
 14  sephora_exclusive   8494 non-null   int64  
 15  highlights          6287 non-null   object 
 16  primar

In [59]:
prod.isnull().sum() #เช็ค missing values

product_id               0
product_name             0
brand_id                 0
brand_name               0
loves_count              0
rating                 278
reviews                278
size                  1631
ingredients              0
price_usd                0
limited_edition          0
new                      0
online_only              0
out_of_stock             0
sephora_exclusive        0
highlights            2207
primary_category         0
secondary_category       0
tertiary_category        0
child_count              0
child_max_price       5740
child_min_price       5740
dtype: int64

In [60]:
prod.duplicated().sum() #เช็คข้อมูลซ้ำ

np.int64(0)